# 00 · Setup and a map of the project

**FindTheObject-CNN** is a small convolutional neural network (CNN) that looks at a 128×128 photo and answers
two questions at once:

1. **Where is the traffic sign?** It outputs a 32×32 *heatmap*: each cell covers 4×4 pixels, and a bright
   cell means "the sign's center is here".
2. **Which sign is it?** It picks one of the 43 German traffic-sign classes from the GTSRB dataset.

The notebooks run from start to finish in this order. Each one builds on the one before it:

| Notebook | What it covers | Project version |
|---|---|---|
| **00_setup** (this one) | checks your Python, data and trained models | |
| **01_synthetic_scenes** | how training images are made: sign photos pasted onto background photos | v1, v1.5b |
| **02_model** | the network: its layers, its two outputs, and where it reads the class | v1, v1.5 |
| **03_train** | the loss, a 1-minute training run, and how to start a full one | all |
| **04_evaluate_synthetic** | scoring a trained model, its hardest classes and its mistakes | all |
| **05_real_photos** | does a model trained on pasted signs work on real road photos? | v2 |
| **06_presence_empty_decoys** | teaching it to say "no sign here" | v3a, v3a.1 |
| **07_inference_demo** | running a trained model on your own image files | |
| **08_diagnostics** | the one-off checks used to test specific worries | |

### How to run a notebook
1. Open it in VS Code. Click **Select Kernel** (top right), then **Python Environments**, then the project's
   `.venv`.
2. Run the cells top to bottom with **Shift+Enter**, or click **Run All**.

### Two kinds of cells
* **Explaining cells** use files already on disk and write nothing (except, sometimes, pictures into
  `outputs/notebooks/`). They run in seconds to a minute.
* **"Run it for real" cells**, at the end of most notebooks, call the project's programs in `scripts/`. Those
  programs build datasets, train models and record results. They are **off by default**. Each notebook has a
  *Settings* cell at the top with flags like `RUN_FOR_REAL = False`; set one to `True` to run that part.
  With a flag off, the cell still prints the exact command it would run.

Everything real lives in `src/findobject/` (the Python package) and `scripts/` (the programs). The notebooks
import that code; they don't copy it. So a number shown here is computed by the same code that produced the
results in `RESULTS.md`.

In [ ]:
# Setup (the same in every notebook). All paths in this project (configs/, data/, outputs/) are relative
# to the repo root, and the scripts expect to run from there, so move this notebook's working folder up
# to the root first. Safe to run twice.
import os
import sys
from pathlib import Path

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists():  # walk up until we reach the folder holding pyproject.toml
    if root == root.parent:
        raise RuntimeError("can't find the repo root: open this notebook from inside FindTheObject-CNN-Project")
    root = root.parent
os.chdir(root)
if str(root / "notebooks") not in sys.path:
    sys.path.insert(0, str(root / "notebooks"))  # lets us import notebooks/nbhelpers.py

from nbhelpers import NB_OUT, ROOT, find_run, read_json, require, run_script
print("working in", ROOT)

## 1. Python, PyTorch and the device

PyTorch can run the network on the Mac's GPU (called **MPS**, Metal Performance Shaders) or on the CPU.
`pick_device("auto")` picks MPS when it's available. Training and scoring both use it.

In [ ]:
# Which Python and PyTorch this notebook runs on, and which device the model will use.
# "mps" is the Apple GPU. "cpu" also works, but training is several times slower on it.
import platform

import torch

from findobject.utils.device import pick_device

print("python", platform.python_version(), "at", sys.executable)
print("torch ", torch.__version__)
print("device", pick_device("auto"))

## 2. Raw downloads

These are the original datasets, downloaded by hand into `data/raw/` (see the *Data* table in `README.md`).
You only need them to **rebuild** datasets (the REBUILD flag in notebooks 01, 05 and 06). Viewing results and
models works without them.

In [ ]:
# Check that each raw download is unzipped where the code looks for it.
# Each entry: what it is -> (path the code reads, zip file it comes from).
RAW = {
    "GTSRB training signs": ("data/raw/gtsrb/GTSRB/Final_Training/Images", "GTSRB_Final_Training_Images.zip"),
    "GTSRB test signs": ("data/raw/gtsrb/GTSRB/Final_Test/Images", "GTSRB_Final_Test_Images.zip"),
    "GTSRB test labels": ("data/raw/gtsrb/GT-final_test.csv", "GTSRB_Final_Test_GT.zip"),
    "COCO background photos": ("data/raw/coco/val2017", "val2017.zip"),
    "COCO annotations": ("data/raw/coco/annotations/instances_val2017.json", "annotations_trainval2017.zip"),
    "GTSDB road photos (v2+)": ("data/raw/gtsdb/FullIJCNN2013", "FullIJCNN2013.zip"),
}
for what, (path, source) in RAW.items():
    status = "ok" if (ROOT / path).exists() else "MISSING"
    print(f"{status:8} {what:25} {path}   (from {source})")

## 3. Built datasets

The scripts turn the raw downloads into `.npz` files in `data/processed/`. An `.npz` is a zip of numpy
arrays: images, class ids, sign boxes and target heatmaps. Those files are what training and scoring read.

| Files | Built by | What they are |
|---|---|---|
| `train`, `val`, `test` | `make_dataset.py` | synthetic scenes: a GTSRB sign pasted onto a COCO photo |
| `real_*`, `hybrid_*` | `make_real_dataset.py` | v2: windows cut from real road photos, and the same signs pasted onto COCO photos |
| `*_empty`, `val_decoy_*`, `real_empty_*` | `make_empty_dataset.py` | v3a: scenes with **no** sign, for measuring false alarms |

**dev** and **val** splits are for looking at results as often as you like. **test** splits are scored *once*
per trained model and then left alone. Re-scoring test while you tweak things would quietly turn it into a
second val set. The scripts enforce this.

In [ ]:
# List each built file with its size on disk. Sizes come from the file system, so nothing is loaded here.
BUILT = {
    "make_dataset": ["train", "val", "test"],
    "make_real_dataset": ["real_dev", "hybrid_dev", "real_test", "hybrid_test"],
    "make_empty_dataset": ["val_empty", "test_empty", "val_decoy_plain", "val_decoy_aug",
                           "real_empty_dev", "real_empty_test"],
}
for script, names in BUILT.items():
    print(f"from scripts/{script}.py:")
    for name in names:
        path = ROOT / "data/processed" / f"{name}.npz"
        size = f"{path.stat().st_size / 1e6:8.1f} MB" if path.exists() else "     MISSING"
        print(f"  {size}  {name}.npz")

## 4. Trained runs

Each training run saves its weights in `outputs/checkpoints/<run_name>-<date>-<time>/`. `best.pt` is the
epoch with the best validation score; `last.pt` is the final epoch. A checkpoint also stores the full config
it was trained with, so we can read back what each run was.

In [ ]:
# Read what each checkpoint recorded about itself: which config, which epoch was best, and its val score.
# "joint" accuracy = the heatmap peak lands inside the sign AND the class is right.
import re

from findobject.training.checkpoint import load_checkpoint_info

VERSION = {"baseline": "v1", "peak": "v1.5", "peak_fresh": "v1.5b, no aug", "peak_fresh_aug": "v1.5b",
           "peak_empty": "v3a", "peak_decoy": "v3a.1"}  # config run_name (seed suffix removed) -> version
print(f"{'run folder':34} {'version':14} seed  best epoch  val joint")
for run in sorted((ROOT / "outputs/checkpoints").iterdir()):
    info = load_checkpoint_info(run / "best.pt")
    name = re.sub(r"_s\d+$", "", info.config["train"]["run_name"])  # peak_decoy_s1 -> peak_decoy
    print(f"{run.name:34} {VERSION.get(name, '?'):14} {info.config['train']['seed']:4}  {info.epoch:10}  "
          f"{info.metrics['joint_acc']:9.4f}")

## 5. Tests

The package has a pytest suite in `tests/`. Running it here works the same as running `pytest` in a
terminal. The notebooks' own test, which runs every notebook from top to bottom, is opt-in because it takes a
few minutes: `pytest -m notebooks`.

In [ ]:
# Run the unit tests (about half a minute). -q prints one dot per passing test.
import subprocess

result = subprocess.run([sys.executable, "-m", "pytest", "-q"], cwd=ROOT, capture_output=True, text=True)
print(result.stdout[-1500:])  # the summary is at the end
if result.returncode:
    raise RuntimeError("some tests failed; read the output above")

**Next:** `01_synthetic_scenes.ipynb`, how the training images are made.